In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2,3'

In [ ]:
import random, math
from transformers import TrainingArguments
from peft import LoraConfig, TaskType

from ranger.utils import common_utils, json_utils, model_utils
from ranger.train.sft_trainer import SftTrainer

In [ ]:
seed = COMMON_CONFIG['seed']
common_utils.set_seed(seed)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/ranger'
data_dir = f'{work_dir}/data'
train_dir = f'{data_dir}/sft'
out_dir = f'{work_dir}/outputs'

dtype = 'bfloat16'
device_map = 'auto'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def load_datas(train_file_path, train_size):
    all_datas = json_utils.load_jsonl(train_file_path)

    random.shuffle(all_datas)

    train_datas = all_datas[:train_size]
    eval_datas = all_datas[train_size:]

    print(f'\n# sft_runner.load_datas() train_datas size : {len(train_datas)}')
    print(f'# sft_runner.load_datas() eval_datas size : {len(eval_datas)}\n')

    json_utils.write_jsonl(train_datas, f'{train_dir}/train_{train_size}.jsonl')
    json_utils.write_jsonl(eval_datas, f'{train_dir}/eval_{len(all_datas)-train_size}.jsonl')

    return train_datas, eval_datas

In [ ]:
def get_peft_config(lora_r,
                    lora_alpha,
                    lora_dropout,
                    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
                    bias='none',
                    task_type=TaskType.CAUSAL_LM):
    
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias=bias,
        task_type=task_type
    )

    return peft_config

In [ ]:
def get_training_args(out_dir,
                      num_epochs,
                      batch_size,
                      accumulation_steps,
                      learning_rate,
                      weight_decay,
                      warmup_ratio,
                      max_grad_norm,
                      save_strategy='steps',
                      save_steps=10,
                      eval_strategy='steps',
                      eval_steps=10,
                      logging_steps=10,
                      lr_scheduler_type='cosine',
                      load_best_model_at_end=True,
                      metric_for_best_model='eval_loss',
                      save_total_limit=5):
    
    training_args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=accumulation_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        # warmup_ratio=warmup_ratio,
        max_grad_norm=max_grad_norm,

        save_strategy=save_strategy,
        save_steps=save_steps,
        eval_strategy=eval_strategy,
        eval_steps=eval_steps,
        logging_steps=logging_steps,

        lr_scheduler_type=lr_scheduler_type,
        load_best_model_at_end=load_best_model_at_end,
        metric_for_best_model=metric_for_best_model,
        save_total_limit=save_total_limit,

        bf16=True,
        do_train=True,
        label_names=['labels'],
        report_to='none'
    )

    return training_args

In [ ]:
train_file_path = f'{train_dir}/selected_train/train_sft_final.jsonl'
train_size = 23000

train_datas, eval_datas = load_datas(train_file_path, train_size)

In [ ]:
# model_names = ['Llama-3.2-3B', 'Qwen2.5-3B', 'Llama-3.1-8B', 'Qwen2.5-7B']
model_names = ['Llama-3.2-3B']

lora_r, lora_alpha, lora_dropout = 64, 128, 0.05
num_epochs, batch_size, accumulation_steps = 3, 1, 128
learning_rate, weight_decay, warmup_ratio, max_grad_norm = 5e-5, 0.01, 0.05, 1.0
save_and_eval_per_epoch, logging_per_epoch = 5, 5

real_batch_size = batch_size * accumulation_steps
steps_per_epoch = math.ceil(len(train_datas) / real_batch_size)
save_and_eval_steps = max(1, steps_per_epoch // save_and_eval_per_epoch)
logging_steps = max(1, steps_per_epoch // logging_per_epoch)

for model_name in model_names:
    model_name_or_path = model_utils.get_model_name_or_path(model_name)

    sft_trainer = SftTrainer(model_name_or_path, max_seq_length, dtype, device_map=device_map)

    peft_config = get_peft_config(lora_r, lora_alpha, lora_dropout)
    sft_trainer.init_model(peft_config)

    save_dir = f'{out_dir}/sft/{model_name}'

    training_args = get_training_args(
        save_dir, num_epochs, batch_size, accumulation_steps,
        learning_rate, weight_decay, warmup_ratio, max_grad_norm,
        save_steps=save_and_eval_steps, eval_steps=save_and_eval_steps, logging_steps=logging_steps
    )

    sft_trainer.set_and_get_sft_dataset(train_datas, eval_datas)
    sft_trainer.train(training_args, early_stopping_patience=3)
    sft_trainer.clear()